In [1]:
%pip install tensorflow numpy scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

from crf_layer import CRF  # make sure crf_layer.py is in the same folder as this notebook

DATA_DIR = "../data/annotated/preprocessed"
MODEL_OUT_DIR = "../models/ner_model"

X_words_train = np.load(f"{DATA_DIR}/X_words_train.npy")
X_chars_train = np.load(f"{DATA_DIR}/X_chars_train.npy")
y_tags_train = np.load(f"{DATA_DIR}/y_tags_train.npy")
X_words_val = np.load(f"{DATA_DIR}/X_words_val.npy")
X_chars_val = np.load(f"{DATA_DIR}/X_chars_val.npy")
y_tags_val = np.load(f"{DATA_DIR}/y_tags_val.npy")

with open(f"{DATA_DIR}/word_vocab.json") as f:
    word_vocab = json.load(f)
with open(f"{DATA_DIR}/char_vocab.json") as f:
    char_vocab = json.load(f)
with open(f"{DATA_DIR}/tag_vocab.json") as f:
    tag_vocab = json.load(f)
with open(f"{DATA_DIR}/preprocessing_config.json") as f:
    config = json.load(f)

MAX_LEN = config["MAX_LEN"]
MAX_WORD_LEN = config["MAX_WORD_LEN"]
NUM_WORDS = config["word_vocab_size"]
NUM_CHARS = config["char_vocab_size"]
NUM_TAGS = config["tag_vocab_size"]

print(f"Train: {X_words_train.shape[0]} sentences | Val: {X_words_val.shape[0]} sentences")
print(f"MAX_LEN={MAX_LEN}, MAX_WORD_LEN={MAX_WORD_LEN}, vocab: words={NUM_WORDS} chars={NUM_CHARS} tags={NUM_TAGS}")

Train: 585 sentences | Val: 104 sentences
MAX_LEN=46, MAX_WORD_LEN=9, vocab: words=1417 chars=78 tags=12


In [3]:
def compute_seq_lengths(X_words):
    return np.sum(X_words != 0, axis=1).astype(np.int32)

seq_lens_train = compute_seq_lengths(X_words_train)
seq_lens_val = compute_seq_lengths(X_words_val)

print("Example sequence lengths (train):", seq_lens_train[:10])

Example sequence lengths (train): [21 12 21 21  7 44 24 11 46 33]


In [4]:
WORD_EMBED_DIM = 100
CHAR_EMBED_DIM = 30
CHAR_LSTM_UNITS = 25
WORD_LSTM_UNITS = 100

# --- Word branch ---
word_input = layers.Input(shape=(MAX_LEN,), name="word_input")
word_embed = layers.Embedding(
    input_dim=NUM_WORDS, output_dim=WORD_EMBED_DIM, mask_zero=True, name="word_embedding"
)(word_input)

# --- Character branch ---
char_input = layers.Input(shape=(MAX_LEN, MAX_WORD_LEN), name="char_input")
char_embed = layers.TimeDistributed(
    layers.Embedding(input_dim=NUM_CHARS, output_dim=CHAR_EMBED_DIM, mask_zero=True),
    name="char_embedding",
)(char_input)
char_lstm = layers.TimeDistributed(
    layers.Bidirectional(layers.LSTM(CHAR_LSTM_UNITS)), name="char_bilstm"
)(char_embed)

# --- Combine + sentence-level BiLSTM ---
combined = layers.Concatenate(name="word_char_concat")([word_embed, char_lstm])
combined = layers.Dropout(0.3)(combined)
bilstm_out = layers.Bidirectional(
    layers.LSTM(WORD_LSTM_UNITS, return_sequences=True), name="sentence_bilstm"
)(combined)
bilstm_out = layers.Dropout(0.3)(bilstm_out)

# Dense projection to per-tag "potentials" (emission scores) before the CRF layer
potentials = layers.Dense(NUM_TAGS, name="emission_scores")(bilstm_out)

base_model = Model(inputs=[word_input, char_input], outputs=potentials, name="bilstm_encoder")
base_model.summary()

Model: "bilstm_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ word_input (InputLayer)       │ (None, 46)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ char_input (InputLayer)       │ (None, 46, 9)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ word_embedding (Embedding)    │ (None, 46, 100)           │         141,700 │ word_input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ char_embedding                │ (None, 46, 9, 30)         │           2,340 │ char_input[0][0]           │
│ (TimeDistributed)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 46)                │               0 │ word_input[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ char_bilstm (TimeDistributed) │ (None, 46, 50)            │          11,200 │ char_embedding[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expand_dims (ExpandDims)      │ (None, 46, 1)             │               0 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ zeros_like (ZerosLike)        │ (None, 46, 100)           │               0 │ word_embedding[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logical_or (LogicalOr)        │ (None, 46, 100)           │               0 │ expand_dims[0][0],         │
│                               │                           │                 │ zeros_like[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ ones_like (OnesLike)          │ (None, 46, 50)            │               0 │ char_bilstm[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ word_char_concat              │ (None, 46, 150)           │               0 │ word_embedding[0][0],      │
│ (Concatenate)                 │                           │                 │ char_bilstm[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 46, 150)           │               0 │ logical_or[0][0],          │
│                               │                           │                 │ ones_like[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 46, 150)           │               0 │ word_char_concat[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ any (Any)                     │ (None, 46)                │               0 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sentence_bilstm               │ (None, 46, 200)           │         200,800 │ dropout[0][0], any[0][0]   │
│ (Bidirectional)               │                           │               

 Total params: 358,452 (1.37 MB)

 Trainable params: 358,452 (1.37 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
crf = CRF(num_tags=NUM_TAGS, name="crf_layer")
# Build the CRF layer's transition weight matrix
crf.build((None, MAX_LEN, NUM_TAGS))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

@tf.function
def train_step(word_batch, char_batch, tag_batch, seq_len_batch):
    with tf.GradientTape() as tape:
        potentials = base_model([word_batch, char_batch], training=True)
        loss = crf.loss(potentials, tag_batch, seq_len_batch)
    trainable_vars = base_model.trainable_variables + [crf.transitions]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))
    return loss

@tf.function
def eval_loss_step(word_batch, char_batch, tag_batch, seq_len_batch):
    potentials = base_model([word_batch, char_batch], training=False)
    return crf.loss(potentials, tag_batch, seq_len_batch)

print("Training step functions ready.")

Training step functions ready.


In [6]:
BATCH_SIZE = 16
EPOCHS = 60
PATIENCE = 8  # stop if val loss doesn't improve for this many epochs

n_train = X_words_train.shape[0]
steps_per_epoch = int(np.ceil(n_train / BATCH_SIZE))

best_val_loss = np.inf
epochs_without_improvement = 0
best_weights = None

history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    # shuffle training data each epoch
    perm = np.random.permutation(n_train)
    epoch_loss = 0.0

    for step in range(steps_per_epoch):
        idx = perm[step * BATCH_SIZE : (step + 1) * BATCH_SIZE]
        loss = train_step(
            tf.constant(X_words_train[idx]),
            tf.constant(X_chars_train[idx]),
            tf.constant(y_tags_train[idx]),
            tf.constant(seq_lens_train[idx]),
        )
        epoch_loss += loss.numpy() * len(idx)

    epoch_loss /= n_train

    val_loss = eval_loss_step(
        tf.constant(X_words_val),
        tf.constant(X_chars_val),
        tf.constant(y_tags_val),
        tf.constant(seq_lens_val),
    ).numpy()

    history["train_loss"].append(epoch_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss
    print(f"Epoch {epoch+1:3d}/{EPOCHS}  train_loss={epoch_loss:.4f}  val_loss={val_loss:.4f}"
          f"{'  <- best so far' if improved else ''}")

    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        best_weights = [w.numpy() for w in base_model.trainable_variables] + [crf.transitions.numpy()]
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs).")
            break

# restore best weights
if best_weights is not None:
    for var, w in zip(base_model.trainable_variables, best_weights[:-1]):
        var.assign(w)
    crf.transitions.assign(best_weights[-1])
    print(f"Restored best weights (val_loss={best_val_loss:.4f}).")

Epoch   1/60  train_loss=28.4222  val_loss=24.2788  <- best so far
Epoch   2/60  train_loss=17.6263  val_loss=12.6569  <- best so far
Epoch   3/60  train_loss=7.7946  val_loss=6.7907  <- best so far
Epoch   4/60  train_loss=4.5832  val_loss=5.1780  <- best so far
Epoch   5/60  train_loss=3.2583  val_loss=4.3460  <- best so far
Epoch   6/60  train_loss=2.6071  val_loss=4.0892  <- best so far
Epoch   7/60  train_loss=2.1469  val_loss=3.6071  <- best so far
Epoch   8/60  train_loss=1.7452  val_loss=3.4189  <- best so far
Epoch   9/60  train_loss=1.5526  val_loss=3.3706  <- best so far
Epoch  10/60  train_loss=1.3167  val_loss=3.1076  <- best so far
Epoch  11/60  train_loss=1.0980  val_loss=2.9665  <- best so far
Epoch  12/60  train_loss=0.9470  val_loss=2.8813  <- best so far
Epoch  13/60  train_loss=0.8387  val_loss=3.0216
Epoch  14/60  train_loss=0.6551  val_loss=2.9055
Epoch  15/60  train_loss=0.6527  val_loss=3.1636
Epoch  16/60  train_loss=0.5461  val_loss=3.0732
Epoch  17/60  train_

In [7]:
inv_tag_vocab = {v: k for k, v in tag_vocab.items()}

val_potentials = base_model([tf.constant(X_words_val), tf.constant(X_chars_val)], training=False)
val_predictions = crf(val_potentials, tf.constant(seq_lens_val)).numpy()

def decode_tags(tag_ids, seq_len):
    return [inv_tag_vocab[i] for i in tag_ids[:seq_len]]

all_true_tags = []
all_pred_tags = []
for i in range(len(X_words_val)):
    seq_len = seq_lens_val[i]
    true_tags = decode_tags(y_tags_val[i], seq_len)
    pred_tags = decode_tags(val_predictions[i], seq_len)
    all_true_tags.append(true_tags)
    all_pred_tags.append(pred_tags)

# quick look at one example
print("Example validation sentence:")
print("True:", all_true_tags[0])
print("Pred:", all_pred_tags[0])

Example validation sentence:
True: ['B-DRUG', 'B-DOSAGE', 'I-DOSAGE', 'B-DOSAGE', 'I-DOSAGE', 'B-DRUG', 'O']
Pred: ['B-DRUG', 'B-DOSAGE', 'I-DOSAGE', 'B-DOSAGE', 'O', 'B-DRUG', 'O']


C:\Users\rohit\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'crf_layer' (of type CRF) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


In [8]:
%pip install seqeval -q

Note: you may need to restart the kernel to use updated packages.


In [9]:
from seqeval.metrics import classification_report, f1_score

print(classification_report(all_true_tags, all_pred_tags, digits=3))
print(f"\nOverall micro F1: {f1_score(all_true_tags, all_pred_tags):.3f}")

              precision    recall  f1-score   support

   DIAGNOSIS      0.636     0.467     0.538        15
      DOSAGE      0.985     0.985     0.985       136
        DRUG      0.903     0.927     0.915       262
    DURATION      0.000     0.000     0.000         8
   FREQUENCY      0.920     0.950     0.935       121

   micro avg      0.911     0.921     0.916       542
   macro avg      0.689     0.666     0.675       542
weighted avg      0.907     0.921     0.913       542


Overall micro F1: 0.916


In [10]:
import os
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

base_model.save(f"{MODEL_OUT_DIR}/bilstm_encoder.keras")
np.save(f"{MODEL_OUT_DIR}/crf_transitions.npy", crf.transitions.numpy())

with open(f"{MODEL_OUT_DIR}/word_vocab.json", "w") as f:
    json.dump(word_vocab, f)
with open(f"{MODEL_OUT_DIR}/char_vocab.json", "w") as f:
    json.dump(char_vocab, f)
with open(f"{MODEL_OUT_DIR}/tag_vocab.json", "w") as f:
    json.dump(tag_vocab, f)
with open(f"{MODEL_OUT_DIR}/preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Model + vocabs saved -> {MODEL_OUT_DIR}/")
print("To reload: load bilstm_encoder.keras, rebuild the CRF layer, "
      "then set crf.transitions from crf_transitions.npy.")

Model + vocabs saved -> ../models/ner_model/
To reload: load bilstm_encoder.keras, rebuild the CRF layer, then set crf.transitions from crf_transitions.npy.
